# 实践项目 03：脑膜瘤 H&E 形态分级

我们将在 Kaggle Notebook 中读取 12 张脑膜瘤 H&E 原图，切分图块，计算核密度评分并形成三档形态等级，再比较颜色基线和小型 CNN。

## 实践任务
1. 找到并读取 12 张原始 H&E 图像
2. 把每张原图切成固定区域并缩放为模型输入
3. 进行 H&E 染色分解并计算核密度评分
4. 依据训练数据的评分分布形成低、中、高三档形态等级
5. 按原图编号划分训练、验证和测试数据
6. 训练颜色统计基线与小型 CNN
7. 输出混淆矩阵和代表性预测图块

## 需要保存的结果
- `task3_original_images.png`
- `task3_data_visualization.png`
- `task3_training_curve.png`
- `task3_prediction_visualization.png`
- `task3_confusion_matrix.png`
- `task3_pytorch_result.json`


In [ ]:
from pathlib import Path
import json, random, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.color import rgb2hed
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT=Path('/kaggle/input'); OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
# 数据目录中同时挂载其他大图时，把关键词改成实际文件夹名；默认空字符串表示搜索全部 input。
DATA_HINT='' 
print('device:',DEVICE)


## 任务 1：找到并读取 12 张原始 H&E 图像

代码优先查找已经解压的 PNG、JPG、JPEG、TIF 和 TIFF；若 Kaggle 数据以 ZIP 形式挂载，则先解压到 working 目录。保留尺寸接近 1536×2048 的原始病理图像。


In [ ]:
extract_dir=OUT/'sdu_neuro_3_images'
extract_dir.mkdir(exist_ok=True)
for z in INPUT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(z) as f: f.extractall(extract_dir)
    except zipfile.BadZipFile:
        pass

candidates=[]
search_roots=[INPUT,extract_dir]
if DATA_HINT:
    hinted=[p for p in INPUT.rglob('*') if p.is_dir() and DATA_HINT.lower() in str(p).lower()]
    if hinted: search_roots=hinted+[extract_dir]
for root in search_roots:
    for ext in ('*.png','*.jpg','*.jpeg','*.tif','*.tiff','*.bmp'):
        candidates.extend(root.rglob(ext))

records=[]
for p in sorted(set(candidates)):
    try:
        with Image.open(p) as im:
            w,h=im.size
        if min(w,h)>=1000 and max(w,h)>=1500:
            records.append((p,w,h))
    except Exception:
        pass

# TODO 1：从 records 中选择 12 张脑膜瘤 H&E 原图
assert len(records)>=12, f'只找到 {len(records)} 张大尺寸图像，请挂载 SDU-Neuro-3 原始图像数据。'
# 优先选择文件名中包含 1–12 编号且尺寸一致的图像；课程目录中只放置这 12 张图时可直接取前 12 张。
records=records[:12]
for r in records: print(r)
originals=[np.asarray(Image.open(p).convert('RGB')) for p,_,_ in records]


In [ ]:
fig,ax=plt.subplots(3,4,figsize=(12,9))
for i,(a,img) in enumerate(zip(ax.ravel(),originals),1):
    a.imshow(img); a.set_title(f'original {i} | {img.shape[1]}×{img.shape[0]}'); a.axis('off')
plt.tight_layout(); plt.savefig(OUT/'task3_original_images.png',dpi=150); plt.show()


## 任务 2：切分图块并计算核密度评分

每张 1536×2048 原图切成不重叠的 256×256 区域，再缩放为 128×128。使用 H&E 染色分解得到苏木精通道，并用其平均强度作为核密度评分。


In [ ]:
PATCH=256; SIZE=128
images=[]; scores=[]; source_ids=[]; coordinates=[]
for sid,img in enumerate(originals):
    h,w=img.shape[:2]
    for y in range(0,h-PATCH+1,PATCH):
        for x in range(0,w-PATCH+1,PATCH):
            tile=img[y:y+PATCH,x:x+PATCH]
            small=np.asarray(Image.fromarray(tile).resize((SIZE,SIZE),Image.Resampling.LANCZOS))
            h_channel=np.clip(rgb2hed(small)[...,0],0,None)
            score=float(h_channel.mean())
            images.append(small); scores.append(score); source_ids.append(sid); coordinates.append((x,y))
images=np.asarray(images,dtype=np.uint8); scores=np.asarray(scores,np.float32); source_ids=np.asarray(source_ids); coordinates=np.asarray(coordinates)
print('images:',images.shape,'scores:',scores.shape,'sources:',np.unique(source_ids))
assert len(images)==576, f'当前得到 {len(images)} 个图块；请确认 12 张原图尺寸为 1536×2048。'


## 任务 3：按原图划分数据并形成三档形态等级


In [ ]:
train_sources=np.arange(0,8); val_sources=np.arange(8,10); test_sources=np.arange(10,12)
split=np.full(len(images),'test',dtype=object)
split[np.isin(source_ids,train_sources)]='train'; split[np.isin(source_ids,val_sources)]='val'
tr=np.where(split=='train')[0]; va=np.where(split=='val')[0]; te=np.where(split=='test')[0]
q1,q2=np.quantile(scores[tr],[1/3,2/3])
labels=np.digitize(scores,[q1,q2]).astype(np.int64)
# TODO 2：输出每个集合的图块数、原图编号和三类数量
summary=None
print('thresholds:',q1,q2); print(summary)


In [ ]:
pick=[]
for cls in range(3):
    ids=np.where(labels==cls)[0]
    pick.extend(ids[np.linspace(0,len(ids)-1,4,dtype=int)])
fig,ax=plt.subplots(3,4,figsize=(9,7))
for a,i in zip(ax.ravel(),pick):
    a.imshow(images[i]); a.set_title(f'level={labels[i]} | score={scores[i]:.3f}'); a.axis('off')
plt.tight_layout(); plt.savefig(OUT/'task3_data_visualization.png',dpi=160); plt.show()


## 任务 4：建立颜色统计基线


In [ ]:
def color_features(x):
    z=x.astype(np.float32)/255.0
    return np.c_[z.mean((1,2)),z.std((1,2))]
X=color_features(images)
baseline=LogisticRegression(max_iter=1200,class_weight='balanced').fit(X[tr],labels[tr])
base_pred=baseline.predict(X[te])
base_f1=f1_score(labels[te],base_pred,average='macro')
print('baseline macro F1:',base_f1)


## 任务 5：补全三分类 CNN


In [ ]:
class PatchDataset(Dataset):
    def __init__(self,idx,augment=False): self.idx=np.asarray(idx); self.augment=augment
    def __len__(self): return len(self.idx)
    def __getitem__(self,k):
        i=int(self.idx[k]); x=images[i].astype(np.float32)/255.0
        if self.augment and random.random()<.5: x=np.fliplr(x).copy()
        return torch.from_numpy(x.transpose(2,0,1)), torch.tensor(labels[i]), i
train_loader=DataLoader(PatchDataset(tr,True),32,shuffle=True,num_workers=2)
val_loader=DataLoader(PatchDataset(va),64,shuffle=False,num_workers=2)
test_loader=DataLoader(PatchDataset(te),64,shuffle=False,num_workers=2)

class PatchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 3：补全三组卷积模块、自适应平均池化和三分类层
        self.features=None
        self.classifier=None
    def forward(self,x):
        return self.classifier(self.features(x).flatten(1))
model=PatchCNN().to(DEVICE)
print(model)


## 任务 6：完成训练与验证


In [ ]:
loss_fn=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
def evaluate(loader):
    model.eval(); ys=[]; ps=[]; losses=[]; ids=[]
    with torch.no_grad():
        for x,y,i in loader:
            logits=model(x.to(DEVICE)); losses.append(loss_fn(logits,y.to(DEVICE)).item())
            ys.extend(y.numpy()); ps.extend(logits.argmax(1).cpu().numpy()); ids.extend(i.numpy())
    return float(np.mean(losses)),accuracy_score(ys,ps),f1_score(ys,ps,average='macro'),np.array(ys),np.array(ps),np.array(ids)

hist=[]; best_state=None; best_f=-1
for epoch in range(5):
    model.train(); train_losses=[]
    for x,y,_ in train_loader:
        # TODO 4：补全清零梯度、前向计算、loss、反向传播和参数更新
        pass
    vl,va_acc,vf,_,_,_=evaluate(val_loader)
    hist.append([np.mean(train_losses),vl,vf]); print(epoch+1,hist[-1])
    if vf>best_f:
        best_f=vf; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best_state)


In [ ]:
h=np.asarray(hist)
fig,ax=plt.subplots(1,2,figsize=(9,3.5)); ax[0].plot(h[:,0],label='train loss');ax[0].plot(h[:,1],label='val loss');ax[0].legend();ax[1].plot(h[:,2],marker='o');ax[1].set_title('validation macro F1');plt.tight_layout();plt.savefig(OUT/'task3_training_curve.png',dpi=160);plt.show()
tl,acc,macro_f1,y_true,y_pred,test_ids=evaluate(test_loader);cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(4.5,4));plt.imshow(cm,cmap='viridis');plt.colorbar();plt.xticks([0,1,2]);plt.yticks([0,1,2]);plt.xlabel('predicted');plt.ylabel('true');plt.tight_layout();plt.savefig(OUT/'task3_confusion_matrix.png',dpi=160);plt.show()
print('test accuracy:',acc,'macro F1:',macro_f1)


## 任务 7：显示错误图块并比较染色扰动


In [ ]:
wrong=np.where(y_true!=y_pred)[0]
show=wrong[:12]
fig,ax=plt.subplots(3,4,figsize=(9,7))
for a,j in zip(ax.ravel(),show):
    i=test_ids[j];a.imshow(images[i]);a.set_title(f'true {labels[i]} → pred {y_pred[j]}');a.axis('off')
plt.tight_layout();plt.savefig(OUT/'task3_prediction_visualization.png',dpi=160);plt.show()

# TODO 5：把测试图像的红通道乘 1.08、蓝通道乘 0.92，重新评价 macro F1
stain_shift_f1=None
result={'original_images':len(originals),'patches':len(images),'train_patches':len(tr),'val_patches':len(va),'test_patches':len(te),'thresholds':[float(q1),float(q2)],'baseline_macro_f1':float(base_f1),'cnn_accuracy':float(acc),'cnn_macro_f1':float(macro_f1),'stain_shift_macro_f1':stain_shift_f1,'seed':SEED}
(OUT/'task3_pytorch_result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8');result


## 结果总结

比较颜色基线与 CNN 的测试表现，观察三档形态等级的主要混淆，并说明染色变化对预测的影响。
